# Week 4 – EDA Deep Dive
## Univariate and Bivariate Analysis of the Ames Housing Dataset

**Course:** 24ADI204 – Data Science & Visualization  
**Team:** Team 8

### Objectives
1. Perform univariate analysis of important numerical and categorical variables.
2. Visualize distributions using histograms, KDE plots, boxplots and count plots.
3. Perform bivariate analysis using scatter plots and grouped comparisons.
4. Study relationships between housing features and `SalePrice`.
5. Use correlation analysis and a heatmap to identify important numerical relationships.
6. Record observations that can guide later project analysis.

> **Data handling:** The notebook first looks for the Week 3 cleaned CSV. If it is not available, it falls back to the Week 3 uncleaned CSV and performs conservative median/mode imputation so that the Week 4 analysis remains reproducible.


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("Libraries loaded successfully.")


## 1. Load the dataset

In [ ]:
# Candidate locations, ordered from preferred cleaned data to fallback data.
candidates = [
    Path("../Week 3/AmesHousing_Uncleaned_Cleaned.csv"),
    Path("../Week 3/AmesHousing_Cleaned.csv"),
    Path("AmesHousing_Uncleaned_Cleaned.csv"),
    Path("AmesHousing_Cleaned.csv"),
    Path("../Week 3/AmesHousing_Uncleaned.csv"),
    Path("AmesHousing_Uncleaned.csv"),
]

data_path = next((p for p in candidates if p.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "Dataset not found. Place the Week 3 dataset in the Week 3 folder "
        "or beside this notebook."
    )

df = pd.read_csv(data_path)
print(f"Loaded: {data_path}")
print(f"Shape: {df.shape}")
display(df.head())


In [ ]:
# If the Week 3 cleaned dataset is not available, perform conservative EDA-safe imputation.
# This does not overwrite the original file.
numeric_cols = df.select_dtypes(include=np.number).columns
categorical_cols = df.select_dtypes(exclude=np.number).columns

missing_before = int(df.isna().sum().sum())

if missing_before > 0:
    for col in numeric_cols:
        df[col] = df[col].fillna(df[col].median())
    for col in categorical_cols:
        mode = df[col].mode(dropna=True)
        fill_value = mode.iloc[0] if not mode.empty else "Unknown"
        df[col] = df[col].fillna(fill_value)

print(f"Missing values before EDA-safe imputation: {missing_before}")
print(f"Missing values after preparation: {int(df.isna().sum().sum())}")


## 2. Dataset overview

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nDescriptive statistics for numerical variables:")
display(df.describe().T)


## 3. Univariate analysis – numerical variables

The following variables are particularly useful for understanding the housing dataset:
- `SalePrice`
- `Gr Liv Area`
- `Overall Qual`
- `Year Built`
- `Total Bsmt SF`
- `Garage Cars`

The analysis examines central tendency, spread, skewness and potential extreme observations.


In [ ]:
selected_numeric = [
    c for c in ["SalePrice", "Gr Liv Area", "Overall Qual",
                "Year Built", "Total Bsmt SF", "Garage Cars"]
    if c in df.columns
]

summary = df[selected_numeric].describe().T
summary["skewness"] = df[selected_numeric].skew()
display(summary)


In [ ]:
# Histograms with KDE
for col in selected_numeric:
    plt.figure(figsize=(10, 6))
    sns.histplot(df[col], kde=True, bins=30)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()


In [ ]:
# Boxplots for numerical variables
for col in selected_numeric:
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=df[col])
    plt.title(f"Boxplot of {col}")
    plt.xlabel(col)
    plt.tight_layout()
    plt.show()


## 4. Univariate analysis – categorical variables

In [ ]:
categorical_candidates = [
    c for c in ["Overall Qual", "Neighborhood", "House Style",
                "Sale Condition", "Sale Type", "MS Zoning"]
    if c in df.columns
]

for col in categorical_candidates:
    counts = df[col].value_counts().head(15)
    plt.figure(figsize=(11, 6))
    sns.barplot(x=counts.values, y=counts.index)
    plt.title(f"Top categories in {col}")
    plt.xlabel("Count")
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()


## 5. Bivariate analysis – numerical vs numerical

In [ ]:
pairs = [
    ("Gr Liv Area", "SalePrice"),
    ("Total Bsmt SF", "SalePrice"),
    ("Year Built", "SalePrice"),
    ("Garage Cars", "SalePrice"),
]

for x, y in pairs:
    if x in df.columns and y in df.columns:
        plt.figure(figsize=(10, 6))
        sns.scatterplot(data=df, x=x, y=y, alpha=0.55)
        plt.title(f"{x} vs {y}")
        plt.tight_layout()
        plt.show()


## 6. Bivariate analysis – categorical vs numerical

In [ ]:
# Sale price by overall quality
if "Overall Qual" in df.columns and "SalePrice" in df.columns:
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=df, x="Overall Qual", y="SalePrice")
    plt.title("SalePrice by Overall Quality")
    plt.xlabel("Overall Quality")
    plt.ylabel("SalePrice")
    plt.tight_layout()
    plt.show()


In [ ]:
# Average SalePrice by Neighborhood
if "Neighborhood" in df.columns and "SalePrice" in df.columns:
    neighborhood_price = (
        df.groupby("Neighborhood")["SalePrice"]
        .mean()
        .sort_values(ascending=False)
    )

    plt.figure(figsize=(12, 8))
    sns.barplot(x=neighborhood_price.values, y=neighborhood_price.index)
    plt.title("Average SalePrice by Neighborhood")
    plt.xlabel("Average SalePrice")
    plt.ylabel("Neighborhood")
    plt.tight_layout()
    plt.show()


In [ ]:
# House style vs SalePrice
if "House Style" in df.columns and "SalePrice" in df.columns:
    plt.figure(figsize=(11, 6))
    sns.boxplot(data=df, x="House Style", y="SalePrice")
    plt.title("SalePrice by House Style")
    plt.xlabel("House Style")
    plt.ylabel("SalePrice")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()


## 7. Correlation analysis

In [ ]:
corr_cols = [
    c for c in [
        "SalePrice", "Overall Qual", "Gr Liv Area", "Total Bsmt SF",
        "Garage Cars", "Garage Area", "1st Flr SF", "Year Built",
        "Year Remod/Add", "Full Bath", "TotRms AbvGrd"
    ] if c in df.columns
]

corr = df[corr_cols].corr()

print("Correlation with SalePrice:")
display(
    corr["SalePrice"]
    .sort_values(ascending=False)
    .to_frame("Correlation with SalePrice")
)


In [ ]:
plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap of Selected Numerical Variables")
plt.tight_layout()
plt.show()


## 8. Focused relationship summary

In [ ]:
# Generate compact numerical summaries that can be quoted in the report.
if "SalePrice" in df.columns:
    print("SalePrice summary")
    print("-" * 40)
    print(f"Mean:   {df['SalePrice'].mean():,.2f}")
    print(f"Median: {df['SalePrice'].median():,.2f}")
    print(f"Std:    {df['SalePrice'].std():,.2f}")
    print(f"Min:    {df['SalePrice'].min():,.2f}")
    print(f"Max:    {df['SalePrice'].max():,.2f}")

if "SalePrice" in df.columns and "Gr Liv Area" in df.columns:
    print("\nPearson correlation: Gr Liv Area vs SalePrice")
    print(f"{df['Gr Liv Area'].corr(df['SalePrice']):.3f}")

if "SalePrice" in df.columns and "Overall Qual" in df.columns:
    print("Pearson correlation: Overall Qual vs SalePrice")
    print(f"{df['Overall Qual'].corr(df['SalePrice']):.3f}")


## 9. Key observations

After executing the notebook, record the main observations here. The following points are the intended interpretation categories:

1. **Distribution:** Describe whether `SalePrice` and other numerical variables are symmetric or skewed.
2. **Spread:** Identify variables with large variation and variables with relatively compact ranges.
3. **Relationships:** Describe the direction and strength of the relationships between `SalePrice` and major numerical features.
4. **Quality effect:** Compare sale-price distributions across `Overall Qual` categories.
5. **Location effect:** Compare average sale prices across neighborhoods.
6. **Outliers:** Mention unusually large or small observations and whether they appear to be valid observations rather than data-entry errors.
7. **Correlation:** Identify the numerical variables with the strongest relationship to `SalePrice`.

These observations should be based on the generated plots and correlation table rather than assumptions.


## 10. Save analysis-ready data and figures

In [ ]:
# Save a copy of the prepared data for reproducibility.
output_dir = Path("week4_outputs")
output_dir.mkdir(exist_ok=True)

prepared_file = output_dir / "AmesHousing_Week4_Prepared.csv"
df.to_csv(prepared_file, index=False)

# Save the correlation table.
corr.to_csv(output_dir / "correlation_matrix.csv")

print(f"Prepared dataset saved to: {prepared_file}")
print("Correlation matrix saved to: week4_outputs/correlation_matrix.csv")


## Conclusion

Week 4 extends the earlier cleaning and initial visualization work into a structured EDA deep dive. The analysis covers univariate distributions, categorical frequencies, numerical relationships, grouped comparisons and correlation analysis. The resulting plots and tables provide evidence for understanding which housing characteristics are associated with variation in `SalePrice`.

**Submission checklist**
- [ ] Run every notebook cell successfully.
- [ ] Verify all plots render correctly.
- [ ] Review the correlation table.
- [ ] Add 4–6 concrete observations to Section 9.
- [ ] Keep the generated `week4_outputs` folder if required by the instructor.
